# Advanced Feature Engineering for Demand Forecasting

This notebook explores sophisticated feature engineering techniques for demand forecasting, using pizza sales as a case study.

**Techniques Covered:**
- **Multi-Level Temporal Encodings:** Capturing hourly, daily, and weekly patterns.
- **External Data Integration:** Incorporating weather and local event data.
- **Competitive Price Sensitivity:** Modeling the impact of price changes.
- **Promotion Lag Effects:** Accounting for the delayed impact of promotions.
- **Stock-Out Impact Modeling:** Simulating the effect of inventory shortages.
- **Automated Feature Importance:** Using SHAP values to understand feature contributions.

## 1. Setup and Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
shap.initjs()

## 2. Data Generation

We'll generate a synthetic dataset that mimics hourly pizza sales, along with external data for weather and local events.

In [ ]:
def generate_pizza_sales_data(start_date='2022-01-01', periods=365*24):
    dates = pd.date_range(start=start_date, periods=periods, freq='H')
    data = pd.DataFrame({'date': dates})
    
    # Time-based patterns
    data['hour'] = data['date'].dt.hour
    data['day_of_week'] = data['date'].dt.dayofweek
    
    # Base sales with seasonality
    hourly_pattern = 20 * np.sin(2 * np.pi * data['hour'] / 24) + 10 * np.cos(4 * np.pi * data['hour'] / 24)
    weekly_pattern = 15 * np.sin(2 * np.pi * data['day_of_week'] / 7)
    base_sales = 50 + hourly_pattern + weekly_pattern + np.random.normal(0, 5, periods)
    
    # Price and promotions
    data['price'] = 12.5 + np.random.normal(0, 0.5, periods)
    data['promotion'] = (np.random.rand(periods) > 0.95).astype(int) # Promotions on 5% of days
    
    # Sales calculation
    price_effect = (data['price'] - 12.5) * -10
    promo_effect = data['promotion'] * 25
    data['sales'] = base_sales + price_effect + promo_effect
    data['sales'] = data['sales'].astype(int).clip(lower=5)
    
    return data.set_index('date').drop(['hour', 'day_of_week'], axis=1)

def generate_external_data(dates):
    # Weather data
    weather = pd.DataFrame({'date': dates})
    weather['temperature'] = 20 + 10 * np.sin(2 * np.pi * weather['date'].dt.dayofyear / 365) + np.random.normal(0, 3, len(dates))
    weather['precipitation'] = (np.random.rand(len(dates)) > 0.8).astype(int) * np.random.uniform(0, 10, len(dates))
    
    # Event data
    events = pd.DataFrame({'date': dates, 'local_event': 0})
    event_days = np.random.choice(events['date'].dt.date.unique(), 20, replace=False)
    events.loc[events['date'].dt.date.isin(event_days), 'local_event'] = 1
    
    return weather.set_index('date'), events.set_index('date')

# Generate and merge data
sales_df = generate_pizza_sales_data()
weather_df, events_df = generate_external_data(sales_df.index)
df = sales_df.join(weather_df).join(events_df)

print(df.head())
df[['sales', 'price', 'temperature']].plot(subplots=True, figsize=(15, 8), layout=(3,1));

## 3. Feature Engineering

In [ ]:
def feature_engineer(df):
    df_eng = df.copy()
    
    # 1. Multi-Level Temporal Encodings
    df_eng['hour'] = df_eng.index.hour
    df_eng['day_of_week'] = df_eng.index.dayofweek
    df_eng['day_of_year'] = df_eng.index.dayofyear
    df_eng['month'] = df_eng.index.month
    df_eng['week_of_year'] = df_eng.index.isocalendar().week.astype(int)
    
    # Cyclical features for hour and month
    df_eng['hour_sin'] = np.sin(2 * np.pi * df_eng['hour'] / 24)
    df_eng['hour_cos'] = np.cos(2 * np.pi * df_eng['hour'] / 24)
    df_eng['month_sin'] = np.sin(2 * np.pi * df_eng['month'] / 12)
    df_eng['month_cos'] = np.cos(2 * np.pi * df_eng['month'] / 12)
    
    # 2. Price Sensitivity Indicator
    df_eng['price_vs_avg'] = df_eng['price'] - df_eng['price'].rolling(30*24).mean()
    
    # 3. Promotion Lag Effects
    # We assume a promotion's effect might linger for a few hours
    df_eng['promo_lag_1'] = df_eng['promotion'].shift(1).fillna(0)
    df_eng['promo_lag_2'] = df_eng['promotion'].shift(2).fillna(0)
    df_eng['promo_rolling_24h'] = df_eng['promotion'].rolling(24).sum().fillna(0)
    
    # 4. Stock-Out Impact Modeling
    # Simulate stock-outs when sales are unusually high for a sustained period
    df_eng['rolling_avg_sales'] = df_eng['sales'].rolling(6).mean().fillna(0)
    high_sales_threshold = df_eng['rolling_avg_sales'].quantile(0.95)
    df_eng['stock_out_risk'] = (df_eng['rolling_avg_sales'] > high_sales_threshold).astype(int)
    
    # Lag this feature as stock-outs impact future sales
    df_eng['stock_out_impact'] = df_eng['stock_out_risk'].shift(1).fillna(0)
    
    # Drop intermediate columns
    df_eng = df_eng.drop(['rolling_avg_sales', 'stock_out_risk'], axis=1)
    
    return df_eng.dropna()

df_engineered = feature_engineer(df)
print("Engineered features:")
print(df_engineered.head())

## 4. Model Training and Evaluation

We will train two models:
1.  **Baseline Model:** Uses a simple set of features.
2.  **Advanced Model:** Uses all the engineered features.

In [ ]:
# Define feature sets
TARGET = 'sales'
BASELINE_FEATURES = ['price', 'promotion', 'temperature', 'precipitation', 'local_event', 'hour', 'day_of_week', 'month']
ADVANCED_FEATURES = [col for col in df_engineered.columns if col != TARGET and col not in ['price', 'promotion']]

# Split data
X = df_engineered.drop(TARGET, axis=1)
y = df_engineered[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train Baseline Model
model_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_base.fit(X_train[BASELINE_FEATURES], y_train)
preds_base = model_base.predict(X_test[BASELINE_FEATURES])

# Train Advanced Model
model_adv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_adv.fit(X_train[ADVANCED_FEATURES], y_train)
preds_adv = model_adv.predict(X_test[ADVANCED_FEATURES])

# Evaluate
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
mae_base = mean_absolute_error(y_test, preds_base)
rmse_adv = np.sqrt(mean_squared_error(y_test, preds_adv))
mae_adv = mean_absolute_error(y_test, preds_adv)

print("--- Baseline Model Performance ---")
print(f"RMSE: {rmse_base:.2f}")
print(f"MAE: {mae_base:.2f}")

print("--- Advanced Model Performance ---")
print(f"RMSE: {rmse_adv:.2f}")
print(f"MAE: {mae_adv:.2f}")

## 5. Feature Importance Analysis with SHAP

Now we'll use SHAP to understand the impact of our engineered features on the model's predictions.

In [ ]:
# Explain the model's predictions using SHAP
explainer = shap.TreeExplainer(model_adv)
shap_values = explainer.shap_values(X_test[ADVANCED_FEATURES])

# Summary plot
shap.summary_plot(shap_values, X_test[ADVANCED_FEATURES], plot_type="bar")

In [ ]:
# More detailed summary plot
shap.summary_plot(shap_values, X_test[ADVANCED_FEATURES])

### Interpreting the SHAP Plots

The plots above show the contribution of each feature to the model's output. 

- The **bar chart** shows the average absolute SHAP value for each feature, indicating its overall importance.
- The **dot plot** (or summary plot) shows the SHAP values for each sample, providing more detail on the direction and magnitude of the feature's effect. For example, you can see how high or low values of a feature impact the prediction.